### Caso não tenha as libs instaladas no Kernel

In [1]:
#%pip install plotly pandas scikit-learn opencv-python
#%pip install --upgrade nbformat

### Import das libs

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cv2
import math
import warnings
import time
import glob
import os
import json
from pathlib import Path
from PIL import Image

from IPython.display import Markdown, display
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### Analise dos logs de voo em memmap

In [6]:
lista_dfs = []
caminhos_ficheiros = []


def listar_logs_voo_memmap():
    candidatos = sorted(Path("../logs").glob("voo_teste_*/manifest.json"))
    if not candidatos:
        candidatos = sorted(Path("logs").glob("voo_teste_*/manifest.json"))
    return candidatos

def carregar_log_voo_memmap(manifest_path):
    manifest_path = Path(manifest_path)

    try:
        texto = manifest_path.read_text(encoding="utf-8").strip()

        if not texto:
            print(f"Ignorando manifest vazio: {manifest_path}")
            return pd.DataFrame()

        manifest = json.loads(texto)

    except JSONDecodeError as e:
        print(f"Ignorando manifest inválido: {manifest_path}")
        print(f"Erro: {e}")
        return pd.DataFrame()

    n = int(manifest.get("num_samples", 0))
    if n <= 0:
        return pd.DataFrame()

    intervalos_path = manifest_path.parent / manifest["arrays"]["flight_intervals"]

    if not intervalos_path.exists():
        print(f"Ignorando run sem memmap: {intervalos_path}")
        return pd.DataFrame()

    intervalos = np.load(intervalos_path, mmap_mode="r")[:n]

    df = pd.DataFrame(intervalos)
    df["run_id"] = manifest_path.parent.name
    df["manifest_path"] = str(manifest_path)

    df["tempo_s"] = df["dt_s"].fillna(0).cumsum()
    df["x_rel"] = df["delta_x_m"].fillna(0).cumsum()
    df["y_rel"] = df["delta_y_m"].fillna(0).cumsum()
    df["z_rel"] = df["delta_z_m"].fillna(0).cumsum()
    df["altitude_rel"] = -df["z_rel"]

    return df

for manifest_path in listar_logs_voo_memmap():
    df_temp = carregar_log_voo_memmap(manifest_path)
    if not df_temp.empty:
        caminhos_ficheiros.append(manifest_path)
        lista_dfs.append(df_temp)

if not lista_dfs:
    display(Markdown("Nenhum log novo em memmap encontrado em `logs/voo_teste_*/manifest.json`."))

#### Deslocamento acumulado relativo

In [7]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    timestamp = caminho.parent.name.replace("voo_teste_", "")
    fig_3d = go.Figure()
    fig_3d.add_trace(go.Scatter3d(x=df["x_rel"], y=df["y_rel"], z=df["altitude_rel"], mode="lines", line=dict(color="royalblue", width=4), name="Deslocamento acumulado"))
    fig_3d.add_trace(go.Scatter3d(x=[0.0], y=[0.0], z=[0.0], mode="markers", marker=dict(color="green", size=6), name="Origem relativa"))
    fig_3d.add_trace(go.Scatter3d(x=[df["x_rel"].iloc[-1]], y=[df["y_rel"].iloc[-1]], z=[df["altitude_rel"].iloc[-1]], mode="markers", marker=dict(color="red", size=6, symbol="x"), name="Fim relativo"))
    fig_3d.update_layout(title=f"Deslocamento acumulado por deltas - Run: {timestamp}", scene=dict(xaxis_title="Delta X acumulado (m)", yaxis_title="Delta Y acumulado (m)", zaxis_title="Delta altitude acumulada (m)", camera=dict(eye=dict(x=1.5, y=1.5, z=0.5))), legend=dict(x=0, y=1), margin=dict(l=0, r=0, b=0, t=40))
    fig_3d.show()

#### Variacao das velocidades angulares

In [8]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    timestamp = caminho.parent.name.replace("voo_teste_", "")
    fig_2d = go.Figure()
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_roll_speed_rad_s"], mode="lines", name="Delta roll speed", opacity=0.7))
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_pitch_speed_rad_s"], mode="lines", name="Delta pitch speed", opacity=0.7))
    fig_2d.add_trace(go.Scatter(x=df["tempo_s"], y=df["delta_yaw_speed_rad_s"], mode="lines", name="Delta yaw speed", opacity=0.7))
    fig_2d.update_layout(title=f"Variacao das velocidades angulares - Run: {timestamp}", xaxis_title="Tempo acumulado por intervalos (s)", yaxis_title="Delta velocidade angular (rad/s)", template="plotly_white", hovermode="x unified")
    fig_2d.show()

#### Analise dos intervalos depth/flow em memmap

Esta analise usa as novas runs em `datasets/depth_ground_truth/run_*/manifest.json`. Cada amostra representa a variacao entre duas atualizacoes consecutivas da logica de proximidade visual por optical flow, a mesma logica que gera as flechas desenhadas na deteccao de obstaculos.

O foco deixa de ser o estado absoluto do drone em um frame isolado e passa a ser o deslocamento sincronizado do intervalo: deltas de posicao, atitude, IMU, comandos reativos, flow e depth ground truth. O objetivo e deixar os dados menos dependentes da origem da simulacao e mais genericos para treino e analise.

In [9]:
# Funcoes para leitura das novas runs depth/flow em numpy.memmap
def localizar_raiz_projeto_memmap(nome_dataset="datasets"):
    candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidato in candidatos:
        if (candidato / nome_dataset / "depth_ground_truth").exists():
            return candidato
    return Path.cwd()


def listar_runs_depth_memmap(base_dir):
    depth_dir = base_dir / "datasets" / "depth_ground_truth"
    return sorted(path.parent for path in depth_dir.glob("run_*/manifest.json"))


def carregar_manifesto_memmap(run_dir):
    with open(run_dir / "manifest.json", encoding="utf-8") as fp:
        return json.load(fp)


def abrir_array_memmap(run_dir, manifesto, chave):
    return np.load(run_dir / manifesto["arrays"][chave], mmap_mode="r")


def carregar_run_depth_memmap(run_dir):
    manifesto = carregar_manifesto_memmap(run_dir)
    n = int(manifesto.get("num_samples", 0))
    intervalos_mm = abrir_array_memmap(run_dir, manifesto, "intervals")
    intervalos = pd.DataFrame.from_records(intervalos_mm[:n]).copy()
    if not intervalos.empty:
        intervalos["run_id"] = run_dir.name
        intervalos["ordem_intervalo"] = np.arange(len(intervalos))
    return {
        "run_dir": run_dir,
        "manifesto": manifesto,
        "intervalos": intervalos,
        "image_delta_bgr": abrir_array_memmap(run_dir, manifesto, "image_delta_bgr")[:n],
        "depth_delta_log": abrir_array_memmap(run_dir, manifesto, "depth_delta_log")[:n],
        "depth_delta_mask": abrir_array_memmap(run_dir, manifesto, "depth_delta_mask")[:n],
        "flow_vectors": abrir_array_memmap(run_dir, manifesto, "flow_vectors")[:n],
    }


def carregar_todas_runs_depth_memmap(base_dir):
    runs = []
    for run_dir in listar_runs_depth_memmap(base_dir):
        try:
            run = carregar_run_depth_memmap(run_dir)
        except Exception as exc:
            print(f"Run ignorada em {run_dir.name}: {exc}")
            continue
        if len(run["intervalos"]) > 0:
            runs.append(run)
    return runs

In [10]:
raiz_projeto = localizar_raiz_projeto_memmap()
runs_depth_memmap = carregar_todas_runs_depth_memmap(raiz_projeto)

if not runs_depth_memmap:
    display(Markdown(
        "Nenhuma run nova em memmap encontrada em `datasets/depth_ground_truth/run_*/manifest.json`. "
        "Colete uma nova run com `save_ground_truth_dataset:=true`."
    ))
else:
    depth_run = runs_depth_memmap[-1]
    depth_df = depth_run["intervalos"].copy()
    manifesto = depth_run["manifesto"]
    print(f"Run analisada: {depth_run['run_dir'].name}")
    print(f"Intervalos depth/flow validos: {len(depth_df)}")
    print(f"Schema: {manifesto.get('schema_version')}")
    colunas_resumo = [
        "dt_s", "depth_age_s", "depth_dt_s", "delta_x_m", "delta_y_m", "delta_z_m",
        "delta_roll_rad", "delta_pitch_rad", "delta_yaw_heading_rad", "flow_valid_points",
        "flow_track_retention_pct", "flow_mag_p90_px", "radial_flow_p90_px",
        "delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_close_5m_pp",
    ]
    display(depth_df[[c for c in colunas_resumo if c in depth_df.columns]].describe().T)
    fig_depth_delta = go.Figure()
    for coluna, nome in [("delta_depth_p10_m", "Delta depth P10"), ("delta_depth_p50_m", "Delta depth P50"), ("delta_depth_close_5m_pp", "Delta pixels < 5 m")]:
        if coluna in depth_df.columns:
            fig_depth_delta.add_trace(go.Scatter(x=depth_df["ordem_intervalo"], y=depth_df[coluna], mode="lines+markers", name=nome))
    fig_depth_delta.update_layout(title="Variacao de profundidade/proximidade por intervalo visual", xaxis_title="Intervalo visual em ordem de coleta", yaxis_title="Delta do intervalo", template="plotly_white", hovermode="x unified")
    fig_depth_delta.show()
    fig_flow_depth = px.scatter(depth_df, x="flow_mag_p90_px", y="delta_depth_close_5m_pp", color="radial_flow_p90_px", size="flow_valid_points", hover_data=["sample_id", "dt_s", "delta_x_m", "delta_yaw_heading_rad"], title="Flow visual x variacao de ocupacao proxima", template="plotly_white")
    fig_flow_depth.show()
    ranking = depth_df.assign(impacto_proximidade=depth_df["delta_depth_close_5m_pp"].abs()).sort_values(["impacto_proximidade", "flow_mag_p90_px"], ascending=False)
    display(Markdown("**Intervalos mais informativos para inspecao/treino:**"))
    display(ranking[["run_id", "sample_id", "dt_s", "flow_valid_points", "flow_mag_p90_px", "radial_flow_p90_px", "delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_close_5m_pp", "delta_x_m", "delta_y_m", "delta_yaw_heading_rad"]].head(12))

Run analisada: run_20260521_175244
Intervalos depth/flow validos: 81
Schema: depth_interval_memmap_v1


,count,mean,std,min,25%,50%,75%,max
dt_s,81.0,0.051012,0.034633,0.032000,0.032000,0.036000,0.064000,0.232000
depth_age_s,81.0,0.056790,0.015433,0.032000,0.036000,0.064000,0.068000,0.068000
depth_dt_s,81.0,0.030420,0.051829,0.000000,0.000000,0.000000,0.064000,0.264000
delta_x_m,42.0,-0.127316,0.477557,-1.306038,-0.322658,-0.023058,0.140340,1.019899
delta_y_m,42.0,0.197929,0.654236,-0.981400,-0.022801,0.149772,0.548155,2.065132
delta_z_m,42.0,-0.006085,0.012177,-0.044487,-0.008596,-0.001980,0.000912,0.010586
delta_roll_rad,81.0,-0.000108,0.021563,-0.140053,0.000000,0.000000,0.000000,0.106935
delta_pitch_rad,81.0,0.001221,0.010524,-0.046808,0.000000,0.000000,0.000000,0.070142
delta_yaw_heading_rad,81.0,0.022147,0.226630,-0.209381,-0.001696,0.000000,0.000000,2.022786
flow_valid_points,81.0,82.234568,9.095702,38.000000,84.000000,84.000000,84.000000,99.000000


**Intervalos mais informativos para inspecao/treino:**

,run_id,sample_id,dt_s,flow_valid_points,flow_mag_p90_px,radial_flow_p90_px,delta_depth_p10_m,delta_depth_p50_m,delta_depth_close_5m_pp,delta_x_m,delta_y_m,delta_yaw_heading_rad
61,run_20260521_175244,62,0.232,80,41.482697,38.405575,0.566921,2.399734,-14.313498,-1.306038,2.065132,-0.017302
67,run_20260521_175244,68,0.068,72,87.648163,85.794121,0.311414,0.609103,-4.012022,-0.006104,0.666206,-0.005187
51,run_20260521_175244,52,0.100,86,21.047468,20.370153,0.120522,1.246979,-3.831791,-0.615916,0.715596,-0.004493
79,run_20260521_175244,80,0.132,94,17.403009,16.845728,0.136290,0.300277,-2.673597,1.019899,-0.921652,-0.012836
45,run_20260521_175244,46,0.100,85,0.100784,0.087414,-0.012964,-0.102242,2.523510,0.000528,-0.000055,0.000028
58,run_20260521_175244,59,0.064,92,62.992615,13.471632,-0.038946,-0.092707,2.513908,-0.284077,0.556007,0.003962
73,run_20260521_175244,74,0.100,95,20.129610,16.674564,-0.097783,0.243785,-1.847388,0.371944,-0.981400,-0.010449
60,run_20260521_175244,61,0.064,72,11.717738,11.489751,-0.067460,-0.109326,-1.412813,-0.278793,0.506180,-0.000519
77,run_20260521_175244,78,0.068,81,75.720909,75.521439,-0.010017,-0.182749,1.249083,0.345819,-0.726500,0.079742
54,run_20260521_175244,55,0.096,74,30.274620,29.461329,-0.001146,-0.130863,-0.991946,-0.631151,0.377031,0.003598


#### Baseline MLP com vetores de variacao

A MLP agora trabalha com features tabulares de deslocamento entre intervalos visuais: deltas de estado, IMU, comandos, estatisticas do optical flow e estatisticas da diferenca de imagem estabilizada. Os alvos tambem sao deltas de depth/proximidade, e nao profundidades absolutas.

A comparacao contra `DummyRegressor(strategy="mean")` continua sendo usada como referencia minima: a MLP so e util se aprender uma relacao melhor que prever a variacao media observada no treino.

In [11]:
TARGETS_MLP_DELTA = ["delta_depth_p10_m", "delta_depth_p50_m", "delta_depth_p90_m", "delta_depth_close_2m_pp", "delta_depth_close_5m_pp", "delta_depth_close_10m_pp"]


def features_image_delta(img):
    arr = np.asarray(img, dtype=np.float32)
    abs_arr = np.abs(arr)
    return {"img_delta_mean": float(arr.mean()), "img_delta_std": float(arr.std()), "img_delta_abs_mean": float(abs_arr.mean()), "img_delta_abs_p90": float(np.percentile(abs_arr, 90))}


def features_flow_vectors(vetores, valid_points):
    n = int(max(0, min(valid_points, len(vetores))))
    if n == 0:
        return {"flow_vec_rel_std": 0.0, "flow_vec_xy_mean": 0.0, "flow_vec_radial_mean": 0.0, "flow_vec_risk_mean": 0.0}
    v = np.asarray(vetores[:n], dtype=np.float32)
    return {"flow_vec_rel_std": float(v[:, :2].std()), "flow_vec_xy_mean": float(v[:, 2:4].mean()), "flow_vec_radial_mean": float(v[:, 4].mean()), "flow_vec_risk_mean": float(v[:, 5].mean())}


def montar_dataset_mlp_delta_depth(base_dir):
    linhas_x, linhas_y, linhas_meta = [], [], []
    for run in carregar_todas_runs_depth_memmap(base_dir):
        df = run["intervalos"].reset_index(drop=True)
        for i, row in df.iterrows():
            if any(col not in row.index or not np.isfinite(row[col]) for col in TARGETS_MLP_DELTA):
                continue
            feats = {}
            for coluna in df.select_dtypes(include=[np.number]).columns:
                if coluna in TARGETS_MLP_DELTA or coluna.startswith("delta_depth_") or coluna in {"delta_valid_px_pct", "sample_id", "ordem_intervalo"}:
                    continue
                valor = row[coluna]
                feats[coluna] = 0.0 if pd.isna(valor) else float(valor)
            feats.update(features_image_delta(run["image_delta_bgr"][i]))
            feats.update(features_flow_vectors(run["flow_vectors"][i], row.get("flow_valid_points", 0)))
            linhas_x.append(feats)
            linhas_y.append({alvo: float(row[alvo]) for alvo in TARGETS_MLP_DELTA})
            linhas_meta.append({"run_id": run["run_dir"].name, "sample_id": int(row.get("sample_id", i + 1)), "ordem_intervalo": int(row.get("ordem_intervalo", i))})
    return pd.DataFrame(linhas_x), pd.DataFrame(linhas_y), pd.DataFrame(linhas_meta)


def separar_intervalos(meta_df, random_state=42):
    n = len(meta_df)
    indices = np.arange(n)
    runs = meta_df["run_id"].dropna().unique() if n else []
    if len(runs) >= 3:
        rng = np.random.default_rng(random_state)
        runs = np.array(runs); rng.shuffle(runs)
        n_train = max(1, int(len(runs) * 0.7)); n_val = max(1, int(len(runs) * 0.15))
        train_runs = set(runs[:n_train]); val_runs = set(runs[n_train:n_train+n_val]); test_runs = set(runs[n_train+n_val:]) or {runs[-1]}
        train_runs -= test_runs
        return {"train": indices[meta_df["run_id"].isin(train_runs).to_numpy()], "val": indices[meta_df["run_id"].isin(val_runs).to_numpy()], "test": indices[meta_df["run_id"].isin(test_runs).to_numpy()], "modo": "por run_id"}
    n_train = max(1, int(n * 0.70)); n_val = max(1, int(n * 0.15))
    return {"train": indices[:n_train], "val": indices[n_train:n_train+n_val], "test": indices[n_train+n_val:], "modo": "temporal por intervalos"}


def avaliar_delta(y_true, y_pred, targets, modelo, split):
    linhas = []
    for i, alvo in enumerate(targets):
        real = np.asarray(y_true[:, i], dtype=float); pred = np.asarray(y_pred[:, i], dtype=float)
        corr = float(np.corrcoef(real, pred)[0, 1]) if len(real) > 1 and np.std(real) > 1e-9 and np.std(pred) > 1e-9 else np.nan
        linhas.append({"modelo": modelo, "split": split, "alvo_delta": alvo, "MAE": float(mean_absolute_error(real, pred)), "RMSE": float(mean_squared_error(real, pred) ** 0.5), "corr": corr})
    return linhas


def treinar_avaliar_mlp_delta_depth(X, y, meta_df):
    targets = list(y.columns); split = separar_intervalos(meta_df)
    if len(split["test"]) == 0: split["test"] = split["val"]
    if len(split["val"]) == 0: split["val"] = split["test"]
    Xv = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    yv = y.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    mlp = TransformedTargetRegressor(regressor=Pipeline([("x_scaler", StandardScaler()), ("mlp", MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu", solver="lbfgs", alpha=0.01, max_iter=2000, random_state=42))]), transformer=StandardScaler())
    dummy = DummyRegressor(strategy="mean")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore"); mlp.fit(Xv[split["train"]], yv[split["train"]])
    dummy.fit(Xv[split["train"]], yv[split["train"]])
    linhas, predicoes = [], {}
    for nome_split, idx in [("validacao", split["val"]), ("teste", split["test"])]:
        pred_mlp = mlp.predict(Xv[idx]); pred_dummy = dummy.predict(Xv[idx])
        predicoes[nome_split] = {"idx": idx, "real": yv[idx], "mlp": pred_mlp, "dummy": pred_dummy}
        linhas.extend(avaliar_delta(yv[idx], pred_mlp, targets, "MLP", nome_split)); linhas.extend(avaliar_delta(yv[idx], pred_dummy, targets, "Media treino", nome_split))
    return mlp, dummy, split, pd.DataFrame(linhas), predicoes


raiz_mlp = localizar_raiz_projeto_memmap()
X_mlp, y_mlp, meta_mlp = montar_dataset_mlp_delta_depth(raiz_mlp)
if len(X_mlp) < 12:
    display(Markdown("Dataset insuficiente para treinar a MLP de deltas. Colete mais intervalos depth/flow em memmap."))
else:
    modelo_mlp_delta, baseline_media_delta, split_mlp, resultados_mlp_delta, predicoes_mlp_delta = treinar_avaliar_mlp_delta_depth(X_mlp, y_mlp, meta_mlp)
    display(Markdown(f"**Dataset MLP delta:** {len(X_mlp)} intervalos, {X_mlp.shape[1]} features de variacao, {y_mlp.shape[1]} alvos delta. Separacao: {split_mlp['modo']}. Treino={len(split_mlp['train'])}, validacao={len(split_mlp['val'])}, teste={len(split_mlp['test'])}."))
    display(resultados_mlp_delta.sort_values(["split", "alvo_delta", "modelo"]).reset_index(drop=True))
    resultados_teste = resultados_mlp_delta[resultados_mlp_delta["split"] == "teste"]
    px.bar(resultados_teste, x="alvo_delta", y="MAE", color="modelo", barmode="group", title="Erro MAE no teste: MLP de deltas x media do treino", template="plotly_white").show()
    alvo_plot = "delta_depth_close_5m_pp" if "delta_depth_close_5m_pp" in y_mlp.columns else y_mlp.columns[0]
    alvo_idx = list(y_mlp.columns).index(alvo_plot); pred_teste = predicoes_mlp_delta["teste"]; meta_teste = meta_mlp.iloc[pred_teste["idx"]].reset_index(drop=True)
    fig_pred = go.Figure()
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["real"][:, alvo_idx], mode="lines+markers", name="real"))
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["mlp"][:, alvo_idx], mode="lines+markers", name="MLP"))
    fig_pred.add_trace(go.Scatter(x=meta_teste.index, y=pred_teste["dummy"][:, alvo_idx], mode="lines", name="media treino"))
    fig_pred.update_layout(title=f"Predicao no teste para {alvo_plot}", xaxis_title="Intervalos de teste em ordem temporal", yaxis_title="Delta do alvo", template="plotly_white", hovermode="x unified"); fig_pred.show()
    display(Markdown("**Intervalos de teste para inspecao:**"))
    display(pd.DataFrame({"run_id": meta_teste["run_id"], "sample_id": meta_teste["sample_id"], f"{alvo_plot}_real": pred_teste["real"][:, alvo_idx], f"{alvo_plot}_mlp": pred_teste["mlp"][:, alvo_idx], f"{alvo_plot}_baseline_media": pred_teste["dummy"][:, alvo_idx]}).head(15))

**Dataset MLP delta:** 205 intervalos, 44 features de variacao, 6 alvos delta. Separacao: por run_id. Treino=128, validacao=41, teste=36.

,modelo,split,alvo_delta,MAE,RMSE,corr
0,MLP,teste,delta_depth_close_10m_pp,2.293913,8.853612,-0.657326
1,Media treino,teste,delta_depth_close_10m_pp,0.593824,1.484924,NaN
2,MLP,teste,delta_depth_close_2m_pp,3.159339,12.853651,-0.936960
3,Media treino,teste,delta_depth_close_2m_pp,1.592609,6.444213,NaN
4,MLP,teste,delta_depth_close_5m_pp,4.409125,19.740300,-0.895862
5,Media treino,teste,delta_depth_close_5m_pp,1.445566,4.530197,NaN
6,MLP,teste,delta_depth_p10_m,0.168922,0.566901,-0.312297
7,Media treino,teste,delta_depth_p10_m,0.097618,0.375058,NaN
8,MLP,teste,delta_depth_p50_m,0.798431,3.655626,-0.924363
9,Media treino,teste,delta_depth_p50_m,0.251893,0.872944,NaN


**Intervalos de teste para inspecao:**

,run_id,sample_id,delta_depth_close_5m_pp_real,delta_depth_close_5m_pp_mlp,delta_depth_close_5m_pp_baseline_media
0,run_20260521_161617,1,0.000000,-0.001010,-0.057707
1,run_20260521_161617,2,0.001225,-0.408444,-0.057707
2,run_20260521_161617,3,0.000418,-0.247860,-0.057707
3,run_20260521_161617,4,0.000000,-0.001010,-0.057707
4,run_20260521_161617,5,0.000000,-0.001207,-0.057707
5,run_20260521_161617,6,0.000000,-0.001010,-0.057707
6,run_20260521_161617,7,0.000000,0.013658,-0.057707
7,run_20260521_161617,8,0.000000,0.053056,-0.057707
8,run_20260521_161617,9,0.259874,0.412389,-0.057707
9,run_20260521_161617,10,3.193541,-0.009065,-0.057707


#### CNN simples para mapa de delta-depth

A CNN deixa de prever depth absoluto. A entrada passa a ser a diferenca entre frames estabilizados (`image_delta_bgr`) e o alvo denso passa a ser `delta_log_depth`, isto e, a variacao do mapa de profundidade em escala logaritmica entre duas atualizacoes consecutivas.

A variante com self-attention no gargalo continua servindo como comparacao: ela testa se relacionar regioes distantes da imagem melhora a estimativa de mudanca de profundidade/proximidade no intervalo visual.

In [12]:
# Variaveis globais e configuracoes da CNN de delta-depth
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset, Subset
except ImportError as exc:
    raise ImportError("Esta celula precisa do PyTorch. Instale com: python3 -m pip install -r requirements.txt") from exc

BATCH_SIZE_CNN = 8
EPOCHS_CNN_DEPTH = 10
LR_CNN_DEPTH = 1e-3

In [13]:
# Funcoes e classes para o modelo CNN de delta-depth
class DepthDeltaMemmapDataset(Dataset):
    def __init__(self, runs):
        self.runs = runs
        self.indices = [(run_i, i) for run_i, run in enumerate(runs) for i in range(len(run["intervalos"]))]
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        run_i, local_i = self.indices[idx]
        run = self.runs[run_i]; row = run["intervalos"].iloc[local_i]
        image_delta = np.asarray(run["image_delta_bgr"][local_i], dtype=np.float32) / 255.0
        image_delta = np.transpose(image_delta, (2, 0, 1))
        depth_delta = np.asarray(run["depth_delta_log"][local_i], dtype=np.float32)
        mask = np.asarray(run["depth_delta_mask"][local_i], dtype=np.float32)
        metrics = np.array([row.get("delta_depth_p10_m", 0.0), row.get("delta_depth_p50_m", 0.0), row.get("delta_depth_close_5m_pp", 0.0) / 100.0], dtype=np.float32)
        return {"image_delta": torch.from_numpy(image_delta), "depth_delta": torch.from_numpy(depth_delta[None, :, :]), "mask": torch.from_numpy(mask[None, :, :]), "metrics": torch.from_numpy(metrics), "run_id": row.get("run_id", run["run_dir"].name), "sample_id": int(row.get("sample_id", local_i + 1))}


def dividir_dataset_delta_por_run(dataset):
    run_ids = [dataset.runs[run_i]["run_dir"].name for run_i, _ in dataset.indices]
    unique_runs = sorted(set(run_ids)); all_idx = np.arange(len(dataset))
    if len(unique_runs) >= 3:
        test_run, val_run = unique_runs[-1], unique_runs[-2]; train_runs = set(unique_runs[:-2])
        train_idx = [i for i, r in enumerate(run_ids) if r in train_runs]
        val_idx = [i for i, r in enumerate(run_ids) if r == val_run]
        test_idx = [i for i, r in enumerate(run_ids) if r == test_run]
        return train_idx, val_idx, test_idx, f"split por run: treino={sorted(train_runs)}, val={val_run}, teste={test_run}"
    n = len(dataset); train_end = max(1, int(n * 0.70)); val_end = max(train_end + 1, int(n * 0.85)) if n >= 3 else train_end
    return all_idx[:train_end].tolist(), all_idx[train_end:val_end].tolist(), all_idx[val_end:].tolist(), "split temporal por haver menos de tres runs"


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__(); self.block = nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.block(x)


class SpatialSelfAttention2d(nn.Module):
    def __init__(self, channels, num_heads=4, dropout=0.05):
        super().__init__(); self.norm = nn.LayerNorm(channels); self.attn = nn.MultiheadAttention(channels, num_heads=num_heads, dropout=dropout, batch_first=True); self.gamma = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        b, c, h, w = x.shape; tokens = x.flatten(2).transpose(1, 2); tokens_norm = self.norm(tokens); attended, _ = self.attn(tokens_norm, tokens_norm, tokens_norm, need_weights=False); tokens = tokens + self.gamma * attended; return tokens.transpose(1, 2).reshape(b, c, h, w)


class DepthDeltaCNNBaseline(nn.Module):
    def __init__(self, usar_atencao=False):
        super().__init__()
        self.encoder = nn.Sequential(ConvBlock(3, 16, 2), ConvBlock(16, 32, 2), ConvBlock(32, 64, 2))
        self.attention = SpatialSelfAttention2d(64) if usar_atencao else nn.Identity()
        self.decoder = nn.Sequential(ConvBlock(64, 64), nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False), ConvBlock(64, 32), nn.Conv2d(32, 1, 3, padding=1))
        self.metric_head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 32), nn.ReLU(inplace=True), nn.Linear(32, 3))
    def forward(self, x):
        features = self.attention(self.encoder(x)); return self.decoder(features), self.metric_head(features)


def masked_smooth_l1_loss(pred, target, mask):
    loss = F.smooth_l1_loss(pred, target, reduction="none") * mask
    return loss.sum() / mask.sum().clamp_min(1.0)


def treinar_uma_epoca_delta_cnn(modelo, loader, optimizer, device):
    modelo.train(); perdas = []
    for batch in loader:
        image_delta = batch["image_delta"].to(device); depth_delta = batch["depth_delta"].to(device); mask = batch["mask"].to(device); metrics = batch["metrics"].to(device)
        optimizer.zero_grad(set_to_none=True); pred_delta, pred_metrics = modelo(image_delta)
        loss = masked_smooth_l1_loss(pred_delta, depth_delta, mask) + 0.2 * F.smooth_l1_loss(pred_metrics, metrics)
        loss.backward(); optimizer.step(); perdas.append(float(loss.detach().cpu()))
    return float(np.mean(perdas)) if perdas else math.nan


def avaliar_delta_cnn(modelo, loader, device):
    modelo.eval(); map_abs, map_sq, p10_abs, p50_abs, close5_abs, infer_times = [], [], [], [], [], []
    with torch.no_grad():
        for batch in loader:
            image_delta = batch["image_delta"].to(device); depth_delta = batch["depth_delta"].to(device); mask = batch["mask"].to(device); metrics = batch["metrics"].to(device)
            start = time.perf_counter(); pred_delta, pred_metrics = modelo(image_delta)
            if device.type == "cuda": torch.cuda.synchronize()
            infer_times.append((time.perf_counter() - start) / max(1, image_delta.shape[0]))
            diff = (pred_delta - depth_delta) * mask; valid_count = mask.sum().clamp_min(1.0)
            map_abs.append(float(diff.abs().sum().cpu() / valid_count.cpu())); map_sq.append(float((diff ** 2).sum().cpu() / valid_count.cpu()))
            metric_diff = (pred_metrics - metrics).abs().cpu().numpy(); p10_abs.append(float(metric_diff[:, 0].mean())); p50_abs.append(float(metric_diff[:, 1].mean())); close5_abs.append(float((metric_diff[:, 2] * 100.0).mean()))
    return {"map_delta_log_mae": float(np.mean(map_abs)) if map_abs else math.nan, "map_delta_log_rmse": float(np.sqrt(np.mean(map_sq))) if map_sq else math.nan, "delta_p10_mae_m": float(np.mean(p10_abs)) if p10_abs else math.nan, "delta_p50_mae_m": float(np.mean(p50_abs)) if p50_abs else math.nan, "delta_close5_mae_pp": float(np.mean(close5_abs)) if close5_abs else math.nan, "inferencia_ms_por_intervalo": float(np.mean(infer_times) * 1000.0) if infer_times else math.nan}


def treinar_modelo_delta_cnn(nome, usar_atencao, train_loader, val_loader, test_loader, device):
    torch.manual_seed(42); modelo = DepthDeltaCNNBaseline(usar_atencao).to(device); optimizer = torch.optim.AdamW(modelo.parameters(), lr=LR_CNN_DEPTH, weight_decay=1e-4); historico = []
    for epoch in range(1, EPOCHS_CNN_DEPTH + 1):
        train_loss = treinar_uma_epoca_delta_cnn(modelo, train_loader, optimizer, device); val_metrics = avaliar_delta_cnn(modelo, val_loader, device)
        historico.append({"modelo": nome, "epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}})
        print(f"{nome} | epoca {epoch:02d}/{EPOCHS_CNN_DEPTH} | loss={train_loss:.4f} | val_delta_log_mae={val_metrics['map_delta_log_mae']:.4f}")
    return modelo, pd.DataFrame(historico), {"modelo": nome, **avaliar_delta_cnn(modelo, test_loader, device)}


def prever_um_exemplo_delta_cnn(modelo, dataset, indice, device):
    modelo.eval(); sample = dataset[indice]
    with torch.no_grad(): pred_delta, _ = modelo(sample["image_delta"].unsqueeze(0).to(device))
    return {"target_delta_log": sample["depth_delta"].squeeze(0).cpu().numpy(), "pred_delta_log": pred_delta.squeeze(0).squeeze(0).cpu().numpy(), "sample_id": sample["sample_id"], "run_id": sample["run_id"]}

In [14]:
raiz_cnn = localizar_raiz_projeto_memmap()
runs_cnn = carregar_todas_runs_depth_memmap(raiz_cnn)
dataset_cnn = DepthDeltaMemmapDataset(runs_cnn)
print(f"Runs memmap validas: {len(runs_cnn)}")
print(f"Intervalos validos encontrados: {len(dataset_cnn)}")

if len(dataset_cnn) < 12:
    display(Markdown("Dataset insuficiente para treinar a CNN de delta-depth. Colete mais intervalos depth/flow."))
else:
    train_idx, val_idx, test_idx, split_desc = dividir_dataset_delta_por_run(dataset_cnn)
    print(split_desc)
    print(f"Treino={len(train_idx)} | Validacao={len(val_idx)} | Teste={len(test_idx)}")
    if len(train_idx) == 0 or len(val_idx) == 0 or len(test_idx) == 0:
        raise ValueError("Nao ha amostras suficientes para treinar/validar/testar a CNN de delta-depth.")
    train_loader_cnn = DataLoader(Subset(dataset_cnn, train_idx), batch_size=BATCH_SIZE_CNN, shuffle=True, num_workers=0)
    val_loader_cnn = DataLoader(Subset(dataset_cnn, val_idx), batch_size=BATCH_SIZE_CNN, shuffle=False, num_workers=0)
    test_loader_cnn = DataLoader(Subset(dataset_cnn, test_idx), batch_size=BATCH_SIZE_CNN, shuffle=False, num_workers=0)
    device_cnn = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Dispositivo PyTorch: {device_cnn}")
    modelo_cnn_sem_atencao, hist_sem_atencao, metricas_sem_atencao = treinar_modelo_delta_cnn("CNN delta sem atencao", False, train_loader_cnn, val_loader_cnn, test_loader_cnn, device_cnn)
    modelo_cnn_com_atencao, hist_com_atencao, metricas_com_atencao = treinar_modelo_delta_cnn("CNN delta com 1 layer de atencao", True, train_loader_cnn, val_loader_cnn, test_loader_cnn, device_cnn)
    metricas_cnn = pd.DataFrame([metricas_sem_atencao, metricas_com_atencao])
    display(metricas_cnn)
    historico_cnn = pd.concat([hist_sem_atencao, hist_com_atencao], ignore_index=True)
    px.line(historico_cnn, x="epoch", y="val_map_delta_log_mae", color="modelo", markers=True, title="Validacao: MAE do mapa delta-log-depth por epoca", labels={"val_map_delta_log_mae": "MAE delta-log-depth", "epoch": "Epoca"}, template="plotly_white").show()
    metricas_long_cnn = metricas_cnn.melt(id_vars="modelo", var_name="metrica", value_name="valor")
    px.bar(metricas_long_cnn, x="metrica", y="valor", color="modelo", barmode="group", title="Teste: CNN em deltas vs CNN em deltas com atencao", template="plotly_white").show()
    indice_exemplo = test_idx[0]
    exemplo_sem = prever_um_exemplo_delta_cnn(modelo_cnn_sem_atencao, dataset_cnn, indice_exemplo, device_cnn)
    exemplo_com = prever_um_exemplo_delta_cnn(modelo_cnn_com_atencao, dataset_cnn, indice_exemplo, device_cnn)
    fig_exemplo_cnn = go.Figure()
    fig_exemplo_cnn.add_trace(go.Heatmap(z=exemplo_sem["target_delta_log"], coloraxis="coloraxis", name="Delta-log GT"))
    fig_exemplo_cnn.add_trace(go.Heatmap(z=exemplo_sem["pred_delta_log"], coloraxis="coloraxis", name="CNN sem atencao", visible=False))
    fig_exemplo_cnn.add_trace(go.Heatmap(z=exemplo_com["pred_delta_log"], coloraxis="coloraxis", name="CNN com atencao", visible=False))
    fig_exemplo_cnn.update_layout(title=f"Exemplo de delta-log-depth - {exemplo_sem['run_id']} / sample {exemplo_sem['sample_id']}", coloraxis={"colorscale": "RdBu", "cmid": 0}, updatemenus=[{"buttons": [{"label": "Delta-log GT", "method": "update", "args": [{"visible": [True, False, False]}]}, {"label": "CNN sem atencao", "method": "update", "args": [{"visible": [False, True, False]}]}, {"label": "CNN com atencao", "method": "update", "args": [{"visible": [False, False, True]}]}]}])
    fig_exemplo_cnn.show()

Runs memmap validas: 4
Intervalos validos encontrados: 205
split por run: treino=['run_20260521_161617', 'run_20260521_174507'], val=run_20260521_174545, teste=run_20260521_175244
Treino=77 | Validacao=47 | Teste=81
Dispositivo PyTorch: cpu


/home/prograf4080/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12070). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/tmp/ipykernel_1960772/3393110397.py:16: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tens

CNN delta sem atencao | epoca 01/10 | loss=0.0810 | val_delta_log_mae=0.0824
CNN delta sem atencao | epoca 02/10 | loss=0.0421 | val_delta_log_mae=0.0748
CNN delta sem atencao | epoca 03/10 | loss=0.0380 | val_delta_log_mae=0.0581
CNN delta sem atencao | epoca 04/10 | loss=0.0375 | val_delta_log_mae=0.0624
CNN delta sem atencao | epoca 05/10 | loss=0.0379 | val_delta_log_mae=0.0638
CNN delta sem atencao | epoca 06/10 | loss=0.0402 | val_delta_log_mae=0.0637
CNN delta sem atencao | epoca 07/10 | loss=0.0379 | val_delta_log_mae=0.0619
CNN delta sem atencao | epoca 08/10 | loss=0.0360 | val_delta_log_mae=0.0610
CNN delta sem atencao | epoca 09/10 | loss=0.0365 | val_delta_log_mae=0.0632
CNN delta sem atencao | epoca 10/10 | loss=0.0354 | val_delta_log_mae=0.0696
CNN delta com 1 layer de atencao | epoca 01/10 | loss=0.0833 | val_delta_log_mae=0.1204
CNN delta com 1 layer de atencao | epoca 02/10 | loss=0.0454 | val_delta_log_mae=0.0828
CNN delta com 1 layer de atencao | epoca 03/10 | loss=

,modelo,map_delta_log_mae,map_delta_log_rmse,delta_p10_mae_m,delta_p50_mae_m,delta_close5_mae_pp,inferencia_ms_por_intervalo
0,CNN delta sem atencao,0.042659,0.108967,0.045267,0.094986,0.890973,0.897586
1,CNN delta com 1 layer de atencao,0.027439,0.105462,0.031881,0.098123,2.319021,0.515387


#### Validacao da sincronizacao entre flow, IMU e depth

A pergunta principal é se os intervalos salvos em memmap estao coerentes: `dt_s`, `depth_age_s`, `depth_dt_s`, vetores de optical flow e deltas de depth devem descrever a mesma janela temporal da logica de proximidade visual.

In [15]:
COLUNAS_SYNC_INTERVALOS = ["dt_s", "depth_age_s", "depth_dt_s", "flow_valid_points", "flow_track_retention_pct", "flow_mag_p90_px", "radial_flow_p90_px", "delta_depth_close_5m_pp", "delta_x_m", "delta_y_m", "delta_yaw_heading_rad", "pan_comp_delta_rad"]


def montar_dataframe_intervalos_memmap(base_dir):
    runs = carregar_todas_runs_depth_memmap(base_dir)
    if not runs:
        return pd.DataFrame()
    partes = []
    for run in runs:
        df = run["intervalos"].copy()
        df["run_id"] = run["run_dir"].name
        partes.append(df)
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


def resumir_sincronizacao_intervalos(df):
    if df.empty:
        return pd.DataFrame()
    cols = [c for c in COLUNAS_SYNC_INTERVALOS if c in df.columns]
    return df[cols].describe().T


raiz_flow = localizar_raiz_projeto_memmap()
intervalos_flow_df = montar_dataframe_intervalos_memmap(raiz_flow)

if intervalos_flow_df.empty:
    display(Markdown("Nenhum intervalo memmap encontrado para validar sincronizacao flow/depth."))
else:
    print(f"Intervalos avaliados: {len(intervalos_flow_df)}")
    print(intervalos_flow_df.groupby("run_id").size().rename("intervalos"))
    display(resumir_sincronizacao_intervalos(intervalos_flow_df))
    px.box(intervalos_flow_df, x="run_id", y="dt_s", title="Distribuicao do intervalo temporal entre atualizacoes visuais", labels={"dt_s": "Delta de tempo do intervalo visual (s)", "run_id": "Run"}, template="plotly_white").show()
    px.scatter(intervalos_flow_df, x="radial_flow_p90_px", y="delta_depth_close_5m_pp", color="run_id", size="flow_valid_points", hover_data=["sample_id", "dt_s", "depth_age_s", "pan_comp_delta_rad"], title="Flow radial do intervalo x variacao de proximidade no depth", labels={"radial_flow_p90_px": "P90 do flow radial (px)", "delta_depth_close_5m_pp": "Delta pixels < 5 m (p.p.)"}, template="plotly_white").show()
    serie = intervalos_flow_df.reset_index(drop=True)
    fig_tempo = go.Figure()
    fig_tempo.add_trace(go.Scatter(x=serie.index, y=serie["flow_mag_p90_px"], mode="lines", name="P90 flow"))
    fig_tempo.add_trace(go.Scatter(x=serie.index, y=serie["delta_depth_close_5m_pp"], mode="lines", name="Delta pixels < 5m", yaxis="y2"))
    fig_tempo.update_layout(title="Sequencia dos intervalos: flow visual e delta de proximidade", xaxis_title="Intervalos concatenados em ordem de leitura", yaxis=dict(title="P90 flow (px)"), yaxis2=dict(title="Delta pixels < 5m (p.p.)", overlaying="y", side="right"), template="plotly_white", hovermode="x unified")
    fig_tempo.show()

Intervalos avaliados: 205
run_id
run_20260521_161617    36
run_20260521_174507    41
run_20260521_174545    47
run_20260521_175244    81
Name: intervalos, dtype: int64


,count,mean,std,min,25%,50%,75%,max
dt_s,205.0,0.055239,0.036619,0.032000,0.032000,0.036000,0.068000,0.232000
depth_age_s,205.0,0.057249,0.014940,0.032000,0.036000,0.064000,0.068000,0.068000
depth_dt_s,205.0,0.034498,0.054008,0.000000,0.000000,0.000000,0.064000,0.264000
flow_valid_points,205.0,82.717073,10.215230,31.000000,80.000000,84.000000,87.000000,119.000000
flow_track_retention_pct,205.0,97.636475,5.830833,55.357143,97.872337,100.000000,100.000000,100.000000
flow_mag_p90_px,205.0,31.688473,47.978779,0.000000,0.097018,13.269078,39.039429,366.418732
radial_flow_p90_px,205.0,26.264482,40.139923,0.000000,0.048331,11.160014,33.409416,332.050354
delta_depth_close_5m_pp,205.0,0.075290,2.976323,-16.466686,0.000000,0.000000,0.000000,24.650038
delta_x_m,165.0,-0.061118,0.443796,-1.306038,-0.323507,-0.000884,0.204542,1.165871
delta_y_m,165.0,0.091821,0.632617,-1.499630,-0.375420,0.062714,0.517101,2.065132
